# Подготовка

In [1]:
IN_GOOGLE_COLAB = get_ipython().__class__.__module__.startswith('google.colab') # type: ignore # 
print(f"{IN_GOOGLE_COLAB=}")

IN_GOOGLE_COLAB=False


In [2]:
!CHCP 65001

Active code page: 65001


## Распаковка архивов

In [3]:
if IN_GOOGLE_COLAB:
    import zipfile
    import os

    # Распаковываем репозиторий
    zip_path = "/content/HandReader-main.zip"
    extract_path = "/content/"

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)

    # Папка после распаковки (обычно HandReader-main)
    repo_folder = "/content/HandReader-main"

    # Если папка существует, переименовываем в handreader для удобства
    if os.path.exists(repo_folder):
        !mv {repo_folder} /content/handreader
        print("✅ Репозиторий распакован в /content/handreader")

In [4]:
if IN_GOOGLE_COLAB:
    # Путь к ZIP с весами (загрузите его тоже через панель Файлы)
    weights_zip = "/content/HandReader_Znaki.zip"

    if os.path.exists(weights_zip):
        with zipfile.ZipFile(weights_zip, 'r') as zip_ref:
            zip_ref.extractall("/content/handreader/weights")
        print("✅ Веса распакованы")
    else:
        print("❌ HandReader_Znaki.zip не найден. Загрузите его в /content/")

## Определение путей

In [5]:
import os
from pathlib import Path
if IN_GOOGLE_COLAB:
    HANDREADER_ROOT = Path(os.path.join("content", "handreader"))
else:
    HANDREADER_ROOT = Path("HandReader")
WEIGHTS_FOLDER = Path("weights") / "znaki"

In [6]:
%pip install -q pickleshare

Note: you may need to restart the kernel to use updated packages.


In [7]:
%cd {HANDREADER_ROOT}

c:\Universitato\Programado\IDA\DactDetect\dactyl-recognition\HandReader


In [8]:
# Проверяем структуру
print("\n📁 Содержимое /content/handreader:")
print(*os.listdir())

print("\n📁 Содержимое configs:")
try:
    print(*os.listdir("configs"))
except:
    print("❌ configs не найдена")

print("\n📁 Содержимое src/models:")
try:
    print(*os.listdir(os.path.join("src", "models")))
except:
    print("❌ src/models не найдена")

print("\n📁 Структура весов:")
try:
    print(*os.listdir(WEIGHTS_FOLDER))
except:
    print(f"❌ {WEIGHTS_FOLDER} не найдена")


📁 Содержимое /content/handreader:
.git .pre-commit-config.yaml .project-root .venv configs data demo_KP KP_reqs.txt pyproject.toml README.md RGB_KP_reqs.txt scripts src tests weights

📁 Содержимое configs:
hydra kp_chicago.yaml kp_rgb_chciago.yaml kp_rgb_Znaki.yaml kp_Znaki.yaml paths rgb_chicago.yaml rgb_Znaki.yaml

📁 Содержимое src/models:
Atester.py Atrainer.py KP KP_RGB modules RGB __init__.py __pycache__

📁 Структура весов:
kp kp_rgb rgb


## Установка библиотек
!!! Требуется `Python 3.9` (но можете попробовать питон выше, вдруг получится)  
!!! При _локальном_ запуске используйте виртуальную среду (`venv`, `conda` и т.п.)  

In [9]:
if IN_GOOGLE_COLAB:
    print("Я не тестил эту ячейку для Гугл-колаба! Уберите эту строчку, если в Колабе всё рабоает")
%pip install typing_extensions==4.13.2
%pip install torch==2.0.1 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cu118
%pip install pyturbojpeg==1.7.7
# conda install conda-forge::pyturbojpeg=1.7.7 
%pip install -r RGB_KP_reqs.txt

Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu118
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


# Импорт модели

In [10]:
# import sys
import torch
# sys.path.append('/content/handreader/src')

from src.models.RGB.models import TSM_Resnet
from src.models.modules.head import RNNHead

# Параметры из чекпоинта:
# - input_dim = 1024 (видно из ошибки: shape torch.Size([3072, 1024]))
# - hidden_dim = 1024
# - num_classes = 35

decoder_net = RNNHead(
    input_dim=1024,          # ← было 512, стало 1024!
    hidden_dim=1024,
    num_layers=2,
    num_classes=35,
    bidirectional=True,
    return_outs=False
)

model = TSM_Resnet(
    encoder='resnet34',
    decoder=None,
    num_classes=35,
    decoder_net=decoder_net,
    unidirection=False
)

# Загружаем веса
checkpoint_path = WEIGHTS_FOLDER / "rgb" / "best.pt"
checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

if 'state_dict' in checkpoint:
    state_dict = checkpoint['state_dict']
else:
    state_dict = checkpoint

model.load_state_dict(state_dict, strict=True)
model.eval()

print("✅ Модель загружена!")

c:\Users\boyko\miniconda3\envs\RGB_KP\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


start FeatureExtracter 22:12:20


c:\Users\boyko\miniconda3\envs\RGB_KP\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\boyko\miniconda3\envs\RGB_KP\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet34_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet34_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


end FeatureExtracter 22:12:21
✅ Модель загружена!


In [11]:
print("📊 Все компоненты TSM_Resnet:\n")
for name, module in model.named_children():
    print(f"  {name}: {module.__class__.__name__}")

print("\n📊 Все подкомпоненты (named_modules):\n")
for name, _ in model.named_modules():
    if name and '.' not in name:  # только верхний уровень
        print(f"  {name}")

📊 Все компоненты TSM_Resnet:

  backbone: FeatureExtracter
  decoder_net: RNNHead

📊 Все подкомпоненты (named_modules):

  backbone
  decoder_net


In [12]:
import torch

class TSAMExtractor(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.backbone = model.backbone

    def forward(self, x, lengths):
        # lengths должен быть тензором, не списком!
        lengths_tensor = torch.tensor(lengths)
        return self.backbone(x, lengths_tensor)

extractor = TSAMExtractor(model)
extractor.eval()

# Тестовый прогон
print("🎯 TSAM-эмбеддинги:\n")

for length in [8, 16, 32]:
    dummy = torch.randn(length, 3, 224, 224)
    lengths = [length]
    with torch.no_grad():
        emb = extractor(dummy, lengths)
    print(f"   Видео из {length:2d} кадров → эмбеддинги: {emb.shape}")

print(f"\n✅ Каждый кадр даёт вектор из {emb.shape[1]} признаков")

🎯 TSAM-эмбеддинги:

   Видео из  8 кадров → эмбеддинги: torch.Size([1, 8, 1024])
   Видео из 16 кадров → эмбеддинги: torch.Size([1, 16, 1024])
   Видео из 32 кадров → эмбеддинги: torch.Size([1, 32, 1024])

✅ Каждый кадр даёт вектор из 32 признаков


# Импорт на ГПУ или ТПУ

In [13]:
# Устанавливаем доступную версию PyTorch
# !pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 --index-url https://download.pytorch.org/whl/cu118

# Проверяем
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
assert torch.cuda.is_available(), "Исполнение дальнейшего кода требует CUDA!"

PyTorch: 2.0.1+cu118
CUDA: False


AssertionError: Исполнение дальнейшего кода требует CUDA!

In [ ]:
# import sys
import torch

# Проверяем версии
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

# Устанавливаем остальные зависимости
# !pip install -q rootutils pytorch-lightning omegaconf hydra-core mediapipe opencv-python-headless pyturbojpeg wandb tensorboard

# Настраиваем пути
# sys.path.insert(0, '/content/handreader')
# sys.path.insert(0, '/content/handreader/src')

# Импортируем
from src.models.RGB.models import TSM_Resnet
from src.models.modules.head import RNNHead

print("✅ Импорт успешен")

# Создаем модель
decoder_net = RNNHead(
    input_dim=1024,
    hidden_dim=1024,
    num_layers=2,
    num_classes=35,
    bidirectional=True,
    return_outs=False
)

model = TSM_Resnet(
    encoder='resnet34',
    decoder=None,
    num_classes=35,
    decoder_net=decoder_net,
    unidirection=False
)

# Загружаем веса
checkpoint_path = WEIGHTS_FOLDER / "rgb" / "best.pt"
checkpoint = torch.load(checkpoint_path, map_location='cuda', weights_only=False)
state_dict = checkpoint['state_dict'] if 'state_dict' in checkpoint else checkpoint
model.load_state_dict(state_dict, strict=True)
model = model.cuda().eval()

print("✅ Модель на GPU!")

# Экстрактор TSAM
class TSAMExtractor(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.backbone = model.backbone
    def forward(self, x, lengths):
        return self.backbone(x.cuda(), torch.tensor(lengths).cuda())

extractor = TSAMExtractor(model)

# Тест
dummy = torch.randn(16, 3, 224, 224)
with torch.no_grad():
    emb = extractor(dummy, [16])

print(f"\n🎯 TSAM-эмбеддинги: {emb.shape}")
print(f"   Первый вектор: {emb[0, :8].cpu().numpy()}")

print("\n✅ ВСЁ РАБОТАЕТ! Можем экспериментировать с дактилемами!")

# Подгрузка видео

In [18]:
if IN_GOOGLE_COLAB:
    # Подключение к Google Диску
    from google.colab import drive
    drive.mount('/content/drive')

    BASE_PATH = Path("/content/drive/MyDrive") # "Мой диск" на Гугл-диске

    # ПУТЬ К ВАШЕЙ ПАПКЕ на Гугл-диске - ИЗМЕНИТЕ ЭТО!
    # По умолчанию: /content/drive/MyDrive/DactDetect
    FOLDER_PATH = BASE_PATH / "DactDetect"  # <- измените если нужно
    # Видео будут доступны по пути:
    # /content/drive/MyDrive/путь_к_видео/
else:
    # Локальная работа
    BASE_PATH = Path("..") # Корень нашего репозитория

    # ПУТЬ К ВАШЕЙ ПАПКЕ локально относительно корня репозитория - ИЗМЕНИТЕ ЭТО!
    # .. = подняться ещё на папку выше
    FOLDER_PATH = BASE_PATH / ".." / "extracted_corpus"

print(FOLDER_PATH)

..\..\extracted_corpus


In [19]:
# Укажите путь к папке DactDetect и проверим содержимое
import os

print(f"📁 Проверяем папку: {FOLDER_PATH}")

if os.path.exists(FOLDER_PATH):
    print("✅ Папка найдена!")

    # Показываем ВСЁ содержимое папки
    files = os.listdir(FOLDER_PATH)
    print(f"\n📋 ВСЕ файлы в папке ({len(files)} шт.):")

    # Сортируем по типу
    excel_files = []
    video_files = []
    other_files = []

    for f in files:
        if f.endswith(('.xlsx', '.xls')):
            excel_files.append(f)
        elif f.endswith(('.webm', '.mp4', '.avi', '.mov', '.mkv')):
            video_files.append(f)
        else:
            other_files.append(f)

    print(f"\n📊 Excel файлы ({len(excel_files)}):")
    if excel_files:
        for f in excel_files:
            print(f"   ✅ {f}")
    else:
        print("   ❌ НЕТ Excel файлов!")

    print(f"\n🎬 Видео файлы ({len(video_files)}):")
    if video_files:
        for f in video_files[:10]:  # покажем первые 10
            print(f"   🎥 {f}")
        if len(video_files) > 10:
            print(f"   ... и еще {len(video_files) - 10}")
    else:
        print("   ❌ НЕТ видео файлов!")

    print(f"\n📄 Другие файлы ({len(other_files)}):")
    for f in other_files[:5]:
        print(f"   📄 {f}")

else:
    print(f"❌ Папка НЕ найдена!")
    print(f"\n🔍 Поиск возможных папок {'на диске' if IN_GOOGLE_COLAB else 'в репозитории'}:")

    # Ищем все папки с похожим названием
    assert os.path.exists(BASE_PATH), f"Корневая папка не найдена: {BASE_PATH}"
    all_folders = [f for f in os.listdir(BASE_PATH)
                    if os.path.isdir(os.path.join(BASE_PATH, f))]

    print(f"\n📁 Папки в корне:")
    dact_folders = []
    for folder in all_folders:
        for keyword in ['dact', 'жест', 'hand', 'corpus', 'model', 'extract', 'detect']:
            if keyword in folder.lower():
                dact_folders.append(folder)
                print(f"   🔍 {folder}")
                break

    if dact_folders:
        print(f"\n💡 Возможно, ваша папка одна из этих:")
        for f in dact_folders:
            print(f"   ➡️ {BASE_PATH}/{f}")
    else:
        print("   (нет папок с ключевыми словами в названии)")
        print("\n📋 Все папки в корне:")
        for f in all_folders[:10]:
            print(f"   📁 {f}")

📁 Проверяем папку: ..\..\extracted_corpus
✅ Папка найдена!

📋 ВСЕ файлы в папке (907 шт.):

📊 Excel файлы (1):
   ✅ values.xlsx

🎬 Видео файлы (904):
   🎥 RSLM-b25-s62_01_парк.webm
   🎥 RSLM-m1-s39_01_птиц.webm
   🎥 RSLM-m1-s39_02_канарейка.webm
   🎥 RSLM-m1-s41_01_вон.webm
   🎥 RSLM-m1-s42_01_кот.webm
   🎥 RSLM-m1-s42_02_канарейка.webm
   🎥 RSLM-m1-s42_03_мусор.webm
   🎥 RSLM-m1-s54-d_01_банка.webm
   🎥 RSLM-m1-s54-d_02_птиц.webm
   🎥 RSLM-m1-s57-d_01_канарейка.webm
   ... и еще 894

📄 Другие файлы (2):
   📄 values.json
   📄 video_urls.json


In [20]:
# ТАБЛИЦА С РАСШИФРОВКАМИ - ИЗМЕНИТЕ ПОД СЕБЯ
SPREADSHEET_FILE = "values.xlsx"
# SPREADSHEET_FILE = "Расшифровки видео.xlsx"
assert os.path.exists(FOLDER_PATH / SPREADSHEET_FILE), "Таблица не существует"

In [23]:
%pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.


In [24]:
# from google.colab import files
import pandas as pd
import os

# Проверяем существование файла
assert os.path.exists(FOLDER_PATH / SPREADSHEET_FILE), "Таблица не существует"

excel_file = FOLDER_PATH / SPREADSHEET_FILE

# Загружаем данные
print(f"\n🔄 Читаем файл: {excel_file}")

# Пробуем разные листы
xls = pd.ExcelFile(excel_file)
print(f"📑 Доступные листы: {xls.sheet_names}")

if 'Корпус' in xls.sheet_names:
    df = pd.read_excel(excel_file, sheet_name='Корпус')
    print(f"✅ Загружен лист 'Корпус': {len(df)} записей")
else:
    # Берем первый лист
    sheet_name = xls.sheet_names[0]
    df = pd.read_excel(excel_file, sheet_name=sheet_name)
    print(f"⚠️ Использую лист '{sheet_name}': {len(df)} записей")

print("\n📋 Первые 5 строк:")
display(df.head())

print("\n📊 Колонки таблицы:")
for i, col in enumerate(df.columns):
    print(f"   {i+1}. '{col}'")

# Сохраняем для следующих ячеек
%store df


🔄 Читаем файл: ..\..\extracted_corpus\values.xlsx
📑 Доступные листы: ['corpus']
⚠️ Использую лист 'corpus': 903 записей

📋 Первые 5 строк:


,файл,расшифровка,текст,информант
0,RSLN-n1-s1_01_раз.webm,раз,RSLN-n1-s1,s1
1,RSLN-n2-s1_01_суши.webm,суши,RSLN-n2-s1,s1
2,RSLN-n2-s1_02_был.webm,был,RSLN-n2-s1,s1
3,RSLN-n2-s2_01_стюардесса.webm,стюардесса,RSLN-n2-s2,s2
4,RSLN-n4-s2_01_чай.webm,чай,RSLN-n4-s2,s2



📊 Колонки таблицы:
   1. 'файл'
   2. 'расшифровка'
   3. 'текст'
   4. 'информант'
Stored 'df' (DataFrame)


In [25]:
# Поиск видео файлов
import os

# Восстанавливаем df
try:
    %store -r df
except:
    pass

if 'df' not in dir():
    print("❌ Данные не загружены!")
else:
    print(f"📁 Папка с видео: {FOLDER_PATH}")

    # Определяем колонку с именами файлов
    possible_file_cols = ['файл', 'file', 'filename', 'video', 'видео']
    file_column = None

    for col in possible_file_cols:
        if col in df.columns:
            file_column = col
            break

    if file_column is None:
        file_column = df.columns[0]
        print(f"⚠️ Использую первую колонку: '{file_column}'")
    else:
        print(f"✅ Колонка с файлами: '{file_column}'")

    # Определяем колонку со словами
    possible_word_cols = ['расшифровка', 'word', 'text', 'слово', 'значение']
    word_column = None

    for col in possible_word_cols:
        if col in df.columns:
            word_column = col
            break

    if word_column is None:
        word_column = df.columns[1] if len(df.columns) > 1 else df.columns[0]
        print(f"⚠️ Использую колонку: '{word_column}'")
    else:
        print(f"✅ Колонка со словами: '{word_column}'")

    # Проверяем существование видео
    found_videos = []
    missing_videos = []

    for idx, row in df.iterrows():
        filename = str(row[file_column])
        if pd.isna(filename) or filename == 'nan':
            continue

        filename = filename.strip()
        video_path = os.path.join(FOLDER_PATH, filename)

        if os.path.exists(video_path):
            word = str(row[word_column]) if not pd.isna(row[word_column]) else "unknown"
            found_videos.append({
                'filename': filename,
                'path': video_path,
                'word': word,
                'index': idx
            })
        else:
            missing_videos.append(filename)

    print(f"\n📊 Результаты:")
    print(f"   ✅ Найдено видео: {len(found_videos)}")
    print(f"   ❌ Отсутствует: {len(missing_videos)}")

    if missing_videos:
        print(f"\n⚠️ Первые 5 отсутствующих:")
        for f in missing_videos[:5]:
            print(f"   - {f}")

    if found_videos:
        print(f"\n✅ Первые 5 найденных:")
        for v in found_videos[:5]:
            print(f"   ✓ {v['filename']} -> {v['word']}")

    # Статистика по словам
    from collections import Counter
    word_counts = Counter(v['word'] for v in found_videos)
    print(f"\n🔤 Уникальных слов: {len(word_counts)}")
    print("\n📊 Топ-10 слов:")
    for word, count in word_counts.most_common(10):
        print(f"   {word}: {count} видео")

    # Сохраняем для следующих ячеек
    %store found_videos

📁 Папка с видео: ..\..\extracted_corpus
✅ Колонка с файлами: 'файл'
✅ Колонка со словами: 'расшифровка'

📊 Результаты:
   ✅ Найдено видео: 903
   ❌ Отсутствует: 0

✅ Первые 5 найденных:
   ✓ RSLN-n1-s1_01_раз.webm -> раз
   ✓ RSLN-n2-s1_01_суши.webm -> суши
   ✓ RSLN-n2-s1_02_был.webm -> был
   ✓ RSLN-n2-s2_01_стюардесса.webm -> стюардесса
   ✓ RSLN-n4-s2_01_чай.webm -> чай

🔤 Уникальных слов: 534

📊 Топ-10 слов:
   уже: 18 видео
   парк: 12 видео
   Оля: 12 видео
   лет: 10 видео
   для: 9 видео
   чай: 8 видео
   эээ: 8 видео
   сад: 8 видео
   Вася: 8 видео
   Алёнка: 8 видео
Stored 'found_videos' (list)


# Извлечение TSAM-эмбеддингов из видео


In [26]:
import cv2
import torch
import numpy as np
from tqdm import tqdm

def extract_tsam_embeddings(video_path, extractor, num_frames=16, img_size=224):
    """
    Извлекает TSAM-эмбеддинги из видео

    Args:
        video_path: путь к видеофайлу
        extractor: экстрактор TSAM (уже создан ранее)
        num_frames: количество кадров для извлечения
        img_size: размер кадра

    Returns:
        embeddings: numpy array [num_frames, 1024]
        success: bool
    """
    cap = cv2.VideoCapture(video_path)
    frames = []

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames == 0:
        cap.release()
        return None, False

    # Равномерно выбираем кадры
    indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)

    # Средние значения и std для нормализации (ImageNet)
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)

    for idx in indices:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if ret:
            # BGR -> RGB
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            # Resize
            frame = cv2.resize(frame, (img_size, img_size))
            # Нормализация
            frame = frame.astype(np.float32) / 255.0
            frame = (frame - mean) / std
            # HWC -> CHW
            frame = np.transpose(frame, (2, 0, 1))
            frames.append(frame)

    cap.release()

    # Если кадров не хватило, добиваем последним
    if len(frames) == 0:
        return None, False

    while len(frames) < num_frames:
        frames.append(frames[-1].copy())

    # Преобразуем в тензор и прогоняем через модель
    video_tensor = torch.tensor(np.stack(frames)).cuda()

    with torch.no_grad():
        embeddings = extractor(video_tensor, [num_frames])

    return embeddings.squeeze(0).cpu().numpy(), True

In [ ]:
print("🎯 Извлекаем TSAM-эмбеддинги для видео...\n")

all_embeddings = {}
video_data = []

for video_info in tqdm(found_videos[:10]):  # начнём с 10 видео для теста
    emb, success = extract_tsam_embeddings(
        video_info['path'],
        extractor,
        num_frames=16
    )

    if success:
        all_embeddings[video_info['filename']] = {
            'embeddings': emb,  # [16, 1024]
            'word': video_info['word'],
            'path': video_info['path']
        }
        video_data.append({
            'filename': video_info['filename'],
            'word': video_info['word'],
            'emb_shape': emb.shape
        })

print(f"\n✅ Успешно обработано: {len(all_embeddings)} видео")

if video_data:
    print("\n📊 Результаты:")
    for v in video_data[:5]:
        print(f"   {v['filename']}: {v['word']} → эмбеддинги {v['emb_shape']}")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

# Выбираем первое видео для анализа
if all_embeddings:
    first_video = list(all_embeddings.keys())[0]
    emb = all_embeddings[first_video]['embeddings']
    word = all_embeddings[first_video]['word']

    print(f"🎬 Видео: {first_video}")
    print(f"📝 Слово: {word}")
    print(f"📐 Эмбеддинги: {emb.shape}")

    # Кластеризация
    n_clusters = min(5, len(word))  # примерно по количеству букв
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
    labels = kmeans.fit_predict(emb)

    print(f"\n🏷️ Кластеры для {emb.shape[0]} кадров: {labels}")

    # PCA визуализация
    pca = PCA(n_components=2)
    emb_2d = pca.fit_transform(emb)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Scatter plot
    scatter = axes[0].scatter(emb_2d[:, 0], emb_2d[:, 1], c=labels, cmap='tab10', s=100)
    axes[0].set_title(f'TSAM-эмбеддинги для слова "{word}"')
    axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    plt.colorbar(scatter, ax=axes[0], ticks=range(n_clusters))

    # Временная динамика
    axes[1].plot(range(len(labels)), labels, 'o-', markersize=10, linewidth=2)
    axes[1].set_xlabel('Кадр (время)')
    axes[1].set_ylabel('Кластер')
    axes[1].set_yticks(range(n_clusters))
    axes[1].set_title(f'Динамика кластеров для "{word}"')
    axes[1].grid(alpha=0.3)

    # Добавляем буквы слова на график
    letters = list(word)
    letter_positions = np.linspace(0, len(labels)-1, len(letters))
    for pos, letter in zip(letter_positions, letters):
        axes[1].axvline(x=pos, color='gray', linestyle='--', alpha=0.5)
        axes[1].text(pos, n_clusters-0.5, letter, ha='center', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()

    print("\n💡 Интерпретация:")
    print("   - Каждый цвет = дактилема (статическая поза или переход)")
    print("   - Вертикальные линии = примерные границы букв в слове")
    print("   - Проверьте, совпадают ли смены кластеров с границами букв!")
else:
    print("❌ Нет обработанных видео")

In [ ]:
# Собираем все эмбеддинги в один массив для глобального анализа
all_embs = []
all_words = []

for filename, data in all_embeddings.items():
    all_embs.append(data['embeddings'])
    all_words.extend([data['word']] * data['embeddings'].shape[0])

# Объединяем
X = np.vstack(all_embs)
print(f"📊 Всего эмбеддингов: {X.shape}")

# Глобальная кластеризация
global_kmeans = KMeans(n_clusters=10, random_state=42, n_init='auto')
global_labels = global_kmeans.fit_predict(X)

# PCA для визуализации
pca = PCA(n_components=2)
X_2d = pca.fit_transform(X)

plt.figure(figsize=(12, 8))
scatter = plt.scatter(X_2d[:, 0], X_2d[:, 1], c=global_labels, cmap='tab10', s=5, alpha=0.6)
plt.colorbar(scatter, ticks=range(10))
plt.title('Глобальные TSAM-кластеры (все видео)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
plt.show()